# D4 - Performance Analytics

Bluestock Mutual Fund Analytics Capstone

In [1]:

from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
DB = ROOT / "data" / "db" / "bluestock_mf.db"

con = sqlite3.connect(DB)

funds = pd.read_sql("SELECT * FROM fund_master", con)
nav = pd.read_sql("SELECT * FROM nav_cleaned", con, parse_dates=["date"])
perf = pd.read_sql("SELECT * FROM scheme_performance", con)

print("Funds:", len(funds))
print("NAV rows:", len(nav))
print("Performance rows:", len(perf))


Funds: 40
NAV rows: 64320
Performance rows: 40


## Performance Metrics

In [2]:

def calculate_metrics(group):
    group = group.sort_values("date").copy()
    returns = group["nav"].pct_change().dropna()

    if len(returns) == 0:
        return pd.Series({
            "CAGR": np.nan,
            "Annual_Return": np.nan,
            "Annual_Volatility": np.nan,
            "Sharpe": np.nan,
            "VaR_95": np.nan,
            "Max_Drawdown": np.nan
        })

    beginning = group["nav"].iloc[0]
    ending = group["nav"].iloc[-1]
    n = len(returns)

    cagr = (ending / beginning) ** (252 / n) - 1
    annual_vol = returns.std() * np.sqrt(252)
    sharpe = cagr / annual_vol if annual_vol != 0 else np.nan

    var95 = -np.percentile(returns, 5)

    wealth = (1 + returns).cumprod()
    running_max = wealth.cummax()
    drawdown = wealth / running_max - 1
    mdd = drawdown.min()

    return pd.Series({
        "CAGR": cagr,
        "Annual_Return": cagr,
        "Annual_Volatility": annual_vol,
        "Sharpe": sharpe,
        "VaR_95": var95,
        "Max_Drawdown": mdd
    })

metrics = nav.groupby("amfi_code").apply(
    calculate_metrics,
    include_groups=False
).reset_index()

metrics = metrics.merge(
    funds[["amfi_code", "scheme_name", "category", "fund_house"]],
    on="amfi_code",
    how="left"
)

metrics["CAGR_pct"] = metrics["CAGR"] * 100
metrics["Volatility_pct"] = metrics["Annual_Volatility"] * 100
metrics["VaR_95_pct"] = metrics["VaR_95"] * 100
metrics["MDD_pct"] = metrics["Max_Drawdown"] * 100

display(
    metrics.sort_values("Sharpe", ascending=False)
    [["scheme_name","category","CAGR_pct","Volatility_pct","Sharpe","VaR_95_pct","MDD_pct"]]
    .head(15)
)


,scheme_name,category,CAGR_pct,Volatility_pct,Sharpe,VaR_95_pct,MDD_pct
27,ICICI Pru Liquid Fund - Regular - Growth,Debt,4.939128,0.459554,10.747650,0.019573,-0.097731
31,Kotak Liquid Fund - Regular - Growth,Debt,4.721448,0.471232,10.019371,0.020381,-0.116293
5,ABSL Liquid Fund - Regular - Growth,Debt,4.446752,0.461861,9.627905,0.021844,-0.162250
34,Mirae Asset Large Cap Fund - Regular - Growth,Equity,20.462186,12.025007,1.701636,1.260740,-11.265729
30,Kotak Flexicap Fund - Regular - Growth,Equity,20.419887,13.454334,1.517718,1.332749,-12.973968
19,SBI Bluechip Fund - Regular Plan - Growth,Equity,17.161430,11.636542,1.474788,1.182684,-15.012385
36,Mirae Asset Tax Saver Fund - Regular - Growth,Equity,21.080438,14.964549,1.408692,1.525905,-16.396743
9,Nippon India Large Cap Fund - Regular - Growth,Equity,16.031199,11.978074,1.338379,1.233347,-17.414075
25,ICICI Pru Midcap Fund - Regular - Growth,Equity,21.635659,16.331149,1.324809,1.622409,-18.188514
38,DSP Midcap Fund - Regular - Growth,Equity,19.576761,15.022898,1.303128,1.586598,-17.248106


## Formula Validation


### CAGR
CAGR = (Ending NAV / Beginning NAV)^(252 / n) - 1

### Sharpe Ratio
Sharpe = Annualised Return / Annualised Volatility

### Historical VaR
VaR(95%) = -5th percentile of daily returns

### Maximum Drawdown
MDD = minimum of (Wealth / Running Maximum) - 1


In [3]:

metrics.to_csv(
    ROOT / "data" / "processed" / "performance_metrics.csv",
    index=False
)

print("Saved performance metrics CSV.")


Saved performance metrics CSV.
